In [ ]:
%load_ext autoreload
%autoreload 2

# 02 — Transform

Unify all regional CSVs from `data/01_raw/` into a single standardised dataset at `data/02_transformed/unified_crime_data.csv`.

## Run the Unification Pipeline

In [ ]:
from gta_crime_data.transform.unify_datasets import unify_datasets

df = unify_datasets()

print(f'Shape: {len(df):,} rows × {len(df.columns)} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()

import pandas as pd

# Date range
df['occurrence_date'] = pd.to_datetime(df['occurrence_date'], errors='coerce')
print(f'Date range: {df["occurrence_date"].min().date()} → {df["occurrence_date"].max().date()}')
print()

# Nulls
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls):
    print('Missing values:')
    for col, n in nulls.items():
        print(f'  {col:25s} {n:>8,}  ({n/len(df)*100:.1f}%)')
else:
    print('No missing values.')
print()

## Filter Invalid Data and Validate Schema
We'll separate rows that are missing required values into an `invalid_data.csv` file, and ensure the remaining valid dataset adheres strictly to our `unified_schema`.

In [ ]:
from gta_crime_data.transform.filter_invalid_incidents import filter_invalid_incidents

df = filter_invalid_incidents(df, verbose=True)

## Validate Remaining

In [ ]:
from gta_crime_data.schemas import unified_schema

unified_schema.validate(df)
print(f"Success! The remaining {len(df):,} rows passed validation.")


### De-duplicate by source_identifier, merging multi-offence rows

In [ ]:
from gta_crime_data.transform.deduplicate_incidents import deduplicate_incidents

df = deduplicate_incidents(df)
df

In [ ]:
pre_2010 = df[df['occurrence_date'] < '2010-01-01']
print(f'Rows before 2010: {len(pre_2010):,} ({len(pre_2010)/len(df)*100:.2f}%)')
print()

# Breakdown by region and year
pre_2010['year'] = pre_2010['occurrence_date'].dt.year
print(pre_2010.groupby(['region', 'year']).size().to_string())
print()

# Which raw files contain the pre-2010 records?
print('Pre-2010 records by source file:')
print(pre_2010.groupby('source_file_name').size().to_string())
print()


In [ ]:
df.info()

## Split by Year

In [ ]:
# Partition data by occurrence year from 2020 to current year
import datetime

# Ensure occurrence_date is datetime and extract year
df['occurrence_date'] = pd.to_datetime(df['occurrence_date'], errors='coerce')
df['year'] = df['occurrence_date'].dt.year

current_year = datetime.datetime.now().year
target_years = range(2020, current_year + 1)

print(f"Partitioning data into yearly files (2020 to {current_year})...")

for year in target_years:
    year_df = df[df['year'] == year].drop(columns=['year'])
    if not year_df.empty:
        # Construct the output filename next to the original
        output_name = f"99_partitioned_year-{year}.csv"
        output_path = os.path.join(os.path.dirname(unified_path), output_name)
        
        year_df.to_csv(output_path, index=False)
        print(f"  → Saved {len(year_df):>6,} rows to {output_name}")
    else:
        print(f"  ! No data found for year {year}")

# Clean up temporary year column
df = df.drop(columns=['year'])
print("\nPartitioning complete.")


## Crime Category Breakdown

In [ ]:
df['mapped_crime_category'].value_counts()

### 'MULTIPLE' Categories by Region

In [ ]:
df[df['mapped_crime_category'] == 'MULTIPLE'].groupby('region').size().sort_values(ascending=False)